# Iowa Liquor Sales: Merchandising Experiment Proposal

**Portfolio refresh of a 2019 Thinkful project**

This notebook revisits the original Iowa Liquor Sales A/B Test Proposal while preserving its core business idea: use historical store and product data to identify a merchandising opportunity, then test whether targeted product promotion increases demand.

The refresh corrects the interpretation of the Iowa data, makes the historical dataset window explicit, improves the data-quality checks, and redesigns the proposal as a true randomized store-level experiment.

> The original 2019 notebook remains unchanged in this folder for provenance. This notebook is a reproducibility and experiment-design refresh, not a claim that the proposed test was actually run.


## Historical data provenance

The original notebook loaded a local snapshot of the State of Iowa Liquor Sales dataset. Preserved notebook output indicates:

- **Coverage:** January 3, 2012 through October 31, 2017
- **Rows:** 12,590,909 (final zero-based index `12,590,908`)
- **Schema:** the classic 24-column Iowa Liquor Sales layout

The current Iowa dataset can receive retrospective corrections. A new API query over the same dates is therefore a **reconstruction of the historical window**, not guaranteed to be byte-for-byte identical to the 2017 snapshot.

Official dataset: https://data.iowa.gov/catalog/dataset/1051


## Important semantic correction

The Iowa data represent **spirits purchases/orders by licensed retailers from the state**, not consumer point-of-sale transactions.

In the source fields:

- `State Bottle Cost` = amount paid by the state per bottle
- `State Bottle Retail` = amount paid by the retailer per bottle
- `Sale (Dollars)` = total amount of the retailer's liquor order
- `Bottles Sold` = bottles ordered by the retailer

The original notebook calculated:

`Sale (Dollars) - State Bottle Cost × Bottles Sold`

and called the result store **Profit**. That is not retailer profit. At most it is a proxy for the state's wholesale gross margin on an order line.

Because the dataset does not contain Casey's consumer revenue or operating costs, this refresh does **not** claim to measure Casey's store profit.


## 1. Setup

The notebook supports either:

1. a local CSV representing the reconstructed historical window, or
2. a chunked download from the public Socrata endpoint.

The full historical window is large (about 12.6 million rows in the preserved snapshot), so keeping the raw CSV outside Git is recommended.


In [ ]:
from pathlib import Path
from urllib.parse import urlencode

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 40)

START_DATE = "2012-01-03"
END_DATE = "2017-10-31"

DATA_PATH = Path("data/iowa_liquor_sales_2012-01-03_to_2017-10-31.csv")
SOCRATA_ENDPOINT = "https://data.iowa.gov/resource/m3tr-qhgy.csv"


## 2. Reconstruct the historical window

The helper below downloads the current public records for the historical date window in chunks. Because Iowa may revise historical records, the row count is reported and compared with the preserved 2017 snapshot rather than assumed to match it exactly.


In [ ]:
def download_iowa_window(
    output_path=DATA_PATH,
    start_date=START_DATE,
    end_date=END_DATE,
    chunk_size=50_000,
):
    output_path.parent.mkdir(parents=True, exist_ok=True)

    # The legacy Socrata view uses `date` for the transaction date.
    where = (
        f"date >= '{start_date}T00:00:00.000' "
        f"AND date <= '{end_date}T23:59:59.999'"
    )

    offset = 0
    first = True

    while True:
        params = {
            "$where": where,
            "$limit": chunk_size,
            "$offset": offset,
            "$order": "date, invoice_and_item_number",
        }
        url = f"{SOCRATA_ENDPOINT}?{urlencode(params)}"
        chunk = pd.read_csv(url, low_memory=False)

        if chunk.empty:
            break

        chunk.to_csv(
            output_path,
            mode="w" if first else "a",
            header=first,
            index=False,
        )

        first = False
        offset += len(chunk)
        print(f"Downloaded {offset:,} rows")

        if len(chunk) < chunk_size:
            break

    return output_path


In [ ]:
# Uncomment only if you need to reconstruct the window from the live API.
# download_iowa_window()

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"{DATA_PATH} was not found. Either place the reconstructed CSV there "
        "or run download_iowa_window()."
    )

raw = pd.read_csv(DATA_PATH, low_memory=False)
print(f"Rows loaded: {len(raw):,}")
print(f"Preserved 2017 snapshot reference: 12,590,909 rows")
raw.head()


## 3. Standardize column names and validate the window

Socrata field names have changed over time. The mapping below accepts both legacy export labels and API-style names where practical.


In [ ]:
def normalize_columns(df):
    rename = {
        "Invoice/Item Number": "invoice_item_number",
        "invoice_and_item_number": "invoice_item_number",
        "Date": "date",
        "Store Number": "store_number",
        "store_number": "store_number",
        "Store Name": "store_name",
        "store_name": "store_name",
        "City": "city",
        "city": "city",
        "Zip Code": "zip_code",
        "zip_code": "zip_code",
        "County": "county",
        "county": "county",
        "Category": "category",
        "category": "category",
        "Category Name": "category_name",
        "category_name": "category_name",
        "Vendor Number": "vendor_number",
        "vendor_number": "vendor_number",
        "Vendor Name": "vendor_name",
        "vendor_name": "vendor_name",
        "Item Number": "item_number",
        "item_number": "item_number",
        "Item Description": "item_description",
        "item_description": "item_description",
        "State Bottle Cost": "state_bottle_cost",
        "state_bottle_cost": "state_bottle_cost",
        "State Bottle Retail": "state_bottle_retail",
        "state_bottle_retail": "state_bottle_retail",
        "Bottles Sold": "bottles_ordered",
        "bottles_sold": "bottles_ordered",
        "Sale (Dollars)": "order_dollars",
        "sale_dollars": "order_dollars",
        "Volume Sold (Liters)": "volume_liters",
        "volume_sold_liters": "volume_liters",
    }
    return df.rename(columns={c: rename[c] for c in df.columns if c in rename}).copy()

df = normalize_columns(raw)
df["date"] = pd.to_datetime(df["date"], errors="coerce")

print("Observed date range:", df["date"].min(), "to", df["date"].max())
print("Duplicate invoice-item IDs:", df["invoice_item_number"].duplicated().sum())


## 4. Numeric cleaning and transaction validation

The original notebook interpreted zero and negative derived values as evidence that stores might be giving products away or losing money. Here they are treated first as **data-quality or correction records**.

A key validation is whether:

`order_dollars ≈ state_bottle_retail × bottles_ordered`

Rows that do not reconcile are flagged rather than used to infer retailer economics.


In [ ]:
numeric_cols = [
    "state_bottle_cost",
    "state_bottle_retail",
    "bottles_ordered",
    "order_dollars",
    "volume_liters",
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = (
            df[col]
            .astype(str)
            .str.replace("$", "", regex=False)
            .str.replace(",", "", regex=False)
        )
        df[col] = pd.to_numeric(df[col], errors="coerce")

df["expected_order_dollars"] = (
    df["state_bottle_retail"] * df["bottles_ordered"]
)

df["order_reconciles"] = np.isclose(
    df["order_dollars"],
    df["expected_order_dollars"],
    rtol=0,
    atol=0.01,
    equal_nan=False,
)

quality_summary = pd.Series({
    "rows": len(df),
    "missing_date": df["date"].isna().sum(),
    "missing_store_number": df["store_number"].isna().sum(),
    "missing_item_number": df["item_number"].isna().sum(),
    "zero_or_negative_bottles": (df["bottles_ordered"] <= 0).sum(),
    "zero_or_negative_order_dollars": (df["order_dollars"] <= 0).sum(),
    "non_reconciling_order_rows": (~df["order_reconciles"]).sum(),
})

quality_summary


For demand analysis, use rows with positive bottle counts and order dollars. The reconciliation flag remains available for sensitivity checks because historical correction practices may create legitimate-looking but internally inconsistent records.


In [ ]:
analysis = df.loc[
    df["date"].notna()
    & df["store_number"].notna()
    & df["item_number"].notna()
    & (df["bottles_ordered"] > 0)
    & (df["order_dollars"] > 0)
].copy()

analysis.shape


## 5. Focus on Casey's stores

The original project noticed that many low values in its derived metric belonged to Casey's General Store and chose that chain for a proposed promotion. The refresh keeps Casey's as the business context but no longer labels stores as profitable or unprofitable.

Stable `store_number` is used as the analytical identifier; store names are retained only for presentation.


In [ ]:
casey = analysis[
    analysis["store_name"].astype(str).str.contains("Casey", case=False, na=False)
].copy()

casey_summary = pd.Series({
    "order_lines": len(casey),
    "stores": casey["store_number"].nunique(),
    "items": casey["item_number"].nunique(),
    "first_date": casey["date"].min(),
    "last_date": casey["date"].max(),
    "bottles_ordered": casey["bottles_ordered"].sum(),
    "order_dollars": casey["order_dollars"].sum(),
})

casey_summary


## 6. Product opportunity

The original notebook grouped by both item number and category name, which allowed the same item to appear more than once when category capitalization changed. This refresh ranks by **stable item number first**.

Two observable demand metrics are shown:

- total bottles ordered
- total order dollars

These are wholesale-order measures, not consumer sell-through.


In [ ]:
item_lookup = (
    casey.sort_values("date")
    .groupby("item_number")
    .agg(
        item_description=("item_description", "last"),
        category_name=("category_name", "last"),
    )
)

product_rank = (
    casey.groupby("item_number")
    .agg(
        bottles_ordered=("bottles_ordered", "sum"),
        order_dollars=("order_dollars", "sum"),
        active_stores=("store_number", "nunique"),
        order_lines=("invoice_item_number", "count"),
    )
    .join(item_lookup)
    .sort_values("bottles_ordered", ascending=False)
)

product_rank.head(15)


In [ ]:
top_products = product_rank.head(10).sort_values("bottles_ordered")

ax = top_products["bottles_ordered"].plot(kind="barh", figsize=(9, 6))
ax.set_title("Top Casey's Items by Bottles Ordered")
ax.set_xlabel("Bottles ordered")
ax.set_ylabel("Item number")
plt.tight_layout()
plt.show()


### Historical comparison with the original notebook

The original notebook hard-coded item numbers `11788`, `11776`, and `35918` after ranking its mislabeled margin metric. The refreshed ranking above may differ because:

1. the metric is corrected to observable demand,
2. item number is the grouping key,
3. a current API reconstruction may include retrospective record corrections.

This is intentional. The goal is to preserve the original analytical question while correcting the method.


In [ ]:
original_candidate_items = [11788, 11776, 35918]

product_rank.loc[
    product_rank.index.astype(str).isin([str(x) for x in original_candidate_items])
]


## 7. Store-week baseline for experiment planning

A merchandising treatment would be assigned at the **store level**, so the experimental dataset should be summarized at the store-week level. Historical order data can estimate baseline activity and variability, but they cannot produce causal treatment effects because no historical randomization occurred.


In [ ]:
casey["week"] = casey["date"].dt.to_period("W").dt.start_time

store_week = (
    casey.groupby(["store_number", "week"])
    .agg(
        bottles_ordered=("bottles_ordered", "sum"),
        order_dollars=("order_dollars", "sum"),
        order_lines=("invoice_item_number", "count"),
    )
    .reset_index()
)

store_baseline = (
    store_week.groupby("store_number")
    .agg(
        active_weeks=("week", "nunique"),
        mean_weekly_bottles=("bottles_ordered", "mean"),
        sd_weekly_bottles=("bottles_ordered", "std"),
        mean_weekly_order_dollars=("order_dollars", "mean"),
    )
)

store_baseline.describe()


## 8. Portfolio-ready A/B test design

### Business question

**Does standardized in-store merchandising for selected liquor products increase product demand at Casey's stores relative to comparable stores without the merchandising treatment?**

### Experimental design

**Population**  
Casey's stores with stable recent ordering activity and enough baseline history.

**Unit of randomization**  
Store. Merchandising is implemented at the store level, so inference should respect that assignment unit.

**Treatment**  
One standardized merchandising treatment applied consistently to a pre-specified set of products. For example: defined shelf placement plus a standardized sign. Do not mix multiple treatment variants into one group unless the experiment is explicitly designed to test them.

**Control**  
Business as usual.

**Randomization**  
50/50 treatment and control, preferably stratified or matched on recent store-level order volume and, if sample size permits, geography.

**Pre-period**  
Approximately 6–8 weeks to characterize baseline demand and estimate variability.

**Test period**  
Approximately 4 weeks, with final duration determined by power analysis and operating constraints.

### Metrics

**Preferred primary metric**  
Consumer POS units of promoted items per store-week.

**Fallback primary metric using Iowa data**  
Bottles ordered of promoted items per store-week, explicitly labeled as a lagging wholesale-demand proxy.

**Secondary metrics**

- promoted-item order dollars per store-week
- total liquor bottles ordered per store-week
- total liquor order dollars per store-week
- order frequency
- non-promoted-item volume, to detect cannibalization

### Hypotheses

For the primary outcome:

- **H0:** Treatment does not change mean promoted-item demand relative to control.
- **H1:** Treatment increases mean promoted-item demand relative to control.

### Analysis

Estimate treatment lift at the store assignment level. A difference-in-differences specification can improve precision by incorporating the pre-period:

`(Treatment post − Treatment pre) − (Control post − Control pre)`

Report the estimated lift with a confidence interval and pre-specified significance level.

### Decision rule

Choose the minimum detectable effect, alpha, target power, test duration, and analysis date **before launch**. Avoid informal early stopping based on whether the first two weeks happen to exceed a percentage threshold.


## 9. Why the redesign is stronger

The original 2019 project showed good business instincts: identify a product-level opportunity, propose a low-cost merchandising intervention, define success criteria, and think about seasonality.

The refreshed version makes those instincts analytically defensible by:

- interpreting the source fields correctly,
- separating wholesale order data from retailer profit and consumer sales,
- validating historical transactions before drawing business conclusions,
- grouping products by stable identifiers,
- defining treatment, control, randomization, and assignment unit,
- separating historical baseline analysis from causal inference,
- using pre-specified metrics and statistical decision rules,
- acknowledging cannibalization and data limitations.

That combination — preserving the original idea while correcting the measurement and causal design — is the purpose of this portfolio refresh.


## 10. Limitations and next steps

1. **Historical snapshot reproducibility:** the exact 2017 file is not bundled, and Iowa may revise old records. The preserved notebook's 12,590,909-row output is the historical checkpoint.
2. **No retailer POS data:** Iowa records wholesale purchases by licensees. Consumer sell-through and retailer profit require Casey's internal data.
3. **Orders are a lagging proxy:** stores may order in batches and carry inventory, so weekly orders can be noisy relative to consumer demand.
4. **Store identity changes:** store names can change; stable store numbers should be used where possible.
5. **Power analysis requires current baseline data:** a real launch should use recent eligible-store outcomes, not only a 2012–2017 historical window.
6. **Causal conclusions require an actual randomized test:** this notebook proposes the experiment; it does not manufacture treatment results from observational history.


## Files in this project

- `Iowa_Liquor_Store_AB_Test_Proposal.ipynb` — preserved original 2019 notebook
- `Iowa_Liquor_Store_AB_Test_Audit.md` — audit trail and historical reconstruction notes
- `Iowa_Liquor_AB_Test_Portfolio.ipynb` — this refreshed portfolio notebook
